In [97]:
import cv2
import numpy as np
import glob

In [15]:
def resizeImg(img):
    height, width = img.shape[:2]
    # new Width of the image
    new_width = 450

    # Finding aspect ration of the image to maintain the height and width of the image
    aspect_ratio = height/width
    
    # New Height
    new_height = int(new_width*aspect_ratio)
    
    resized_img = cv2.resize(img, (new_width, new_height), cv2.INTER_LANCZOS4)
    return resized_img

# Practice

In [91]:
# CyberPunk 2077
img1_query = cv2.imread('cyber.JPG', cv2.IMREAD_GRAYSCALE)
img1_train = cv2.imread('cyber2.jpeg', cv2.IMREAD_GRAYSCALE)

# Breath of the wild
img2_query = cv2.imread('botw.jpeg', cv2.IMREAD_GRAYSCALE)
img2_train = cv2.imread('botw2.JPG', cv2.IMREAD_GRAYSCALE)

# Tears of the Kingdom
img3_query = cv2.imread('totk.JPG', cv2.IMREAD_GRAYSCALE)
img3_train = cv2.imread('totk2.JPG', cv2.IMREAD_GRAYSCALE)

# Resizeing images to be of same height and width
cyber_query = resizeImg(img1_query)
cyber_train = resizeImg(img1_train)

In [79]:
# Creating ORB Object
orb = cv2.ORB_create(nfeatures = 1000) # by default nfeatures is 500

# Detecting and computing Keypoints and descriptor of img
kp1, desc1 = orb.detectAndCompute(cyber_query, None)
kp2, desc2 = orb.detectAndCompute(cyber_train, None)

In [81]:
# Drawing the keypoints
imgkp1 = cv2.drawKeypoints(cyber_query, kp1, None)
imgkp2 = cv2.drawKeypoints(cyber_train, kp2, None)

In [93]:
# BruteForce Matcher Object
bf = cv2.BFMatcher()
matches = bf.knnMatch(desc1, desc2, k=2)

# Deciding good match based on the distance
goodMatch = []
for m,n in matches: # unpacking 2 values because I gave k=2 in .knnMatch()
    if m.distance < .75 * n.distance:
        goodMatch.append([m])

In [85]:
len(goodMatch)

85

In [87]:
matchesDetected = cv2.drawMatchesKnn(cyber_query, kp1, cyber_train, kp2, goodMatch, None, flags = 2)

In [89]:
# Showing Images
# cv2.imshow('CyberPunk Query', cyber_query)
# cv2.imshow('CyberPunk Train', cyber_train)

# Showing Keypoints
cv2.imshow('CyberPunk Query', imgkp1)
cv2.imshow('CyberPunk Train', imgkp2)

# Detected Matches
cv2.imshow("Matches Detected", matchesDetected)

cv2.waitKey(0)

-1

# Project 1

In [237]:
def descriptor(images):
    descriptor = []

    # Rotating through images and creating list of descriptor
    for img in images:
        kp, desc = orb.detectAndCompute(img, None)
        descriptor.append(desc)

    return descriptor

In [227]:
def findId(img, descriptors, thresh = 15):
    # finding keypoints and descriptors of training images
    kp2, desc2 = orb.detectAndCompute(img, None)

    # creating BFMatcher object
    bf = cv2.BFMatcher()
    matcheList = []
    finalVal = -1
    
    try:
        for desc in descriptors:
            # finding match b/w query descripter and train descripter
            matches = bf.knnMatch(desc, desc2, k=2)
            goodmatch = []

            # Filtering matches
            for m,n in matches:
                if m.distance < 0.75 * n.distance:
                    goodmatch.append([m])
    
            matcheList.append(len(goodmatch))
    except:
        pass
    
    if len(matcheList) != 0:
        # Compairing Threshold and the maximum of matcheList
        if max(matcheList) > thresh:
            # finding index of the of the best match 
            finalVal = matcheList.index(max(matcheList))
    
    return finalVal

In [229]:
queryImg_path = glob.glob('query_Image/*')
trainImg_path = glob.glob('train_image/*')

query_images = []
train_images = []
queryImg_name = []
trainImg_name = []

orb = cv2.ORB_create(nfeatures = 1000)

for path in queryImg_path:
    currImg = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    resizedImg = resizeImg(currImg)
    query_images.append(resizedImg)
    
    file_name = path.split('\\')[-1]
    queryImg_name.append(file_name.split('.')[0])

for path in trainImg_path:
    currImg = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    resizedImg = resizeImg(currImg)
    train_images.append(resizedImg)
    
    file_name = path.split('\\')[-1]
    trainImg_name.append(file_name.split('.')[0])

In [231]:
descriptors = descriptor(query_images)

In [235]:
for img in train_images:
    img_id = findId(img, descriptors, 10)

# cap = cv2.VideoCapture(0)

# while True:
#     _, frame = cap.read()
#     grey_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

#     img_id = findId(grey_frame, descriptors)

0
0
1
1
-1
2


In [207]:
for img in query_images:
    cv2.imshow('img', img)
    cv2.waitKey(0)